# Face Recognition System 

In [2]:
import os
import numpy as np
from PIL import Image

## Dataset load

In [3]:
dataset_path = "dataset"

images = []
labels = []

for person_name in os.listdir(dataset_path):

    if person_name == ".ipynb_checkpoints":
        continue

    person_path = os.path.join(dataset_path, person_name)

    for img_name in os.listdir(person_path):

        if img_name == ".ipynb_checkpoints":
            continue

        img_path = os.path.join(person_path, img_name)

        image = Image.open(img_path).convert('L')
        image = image.resize((32,32))
        image = np.array(image).flatten()

        images.append(image)
        labels.append(person_name)

X = np.array(images)

## Labels Convert

In [4]:
unique_labels = list(set(labels))

label_to_num = {}

for i, label in enumerate(unique_labels):
    label_to_num[label] = i

y = np.array([label_to_num[label] for label in labels])

num_to_label = {v:k for k,v in label_to_num.items()}

In [5]:
X = X / 255.
0

## MLP Structure

In [6]:
input_size = X.shape[1]
hidden_size = 64
output_size = len(np.unique(y))

np.random.seed(42)

W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

## Functions

In [7]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp = np.exp(x - np.max(x))
    return exp / np.sum(exp, axis=1, keepdims=True)

def forward(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2

def loss(y_true, y_pred):
    m = y_true.shape[0]
    log_likelihood = -np.log(y_pred[range(m), y_true])
    return np.sum(log_likelihood) / m

def one_hot(y, num_classes):
    return np.eye(num_classes)[y]

## One Hot

In [8]:
y_onehot = one_hot(y, output_size)

## Training

In [9]:
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):

    Z1, A1, Z2, A2 = forward(X)

    m = X.shape[0]
#backward
    dZ2 = A2 - y_onehot
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * (A1 > 0)

    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", loss(y, A2))

Epoch: 0 Loss: 1.7906404886253025
Epoch: 100 Loss: 1.730127916601815
Epoch: 200 Loss: 1.5055383445019692
Epoch: 300 Loss: 1.0788747786145854
Epoch: 400 Loss: 0.6907809619961632
Epoch: 500 Loss: 0.4256359337625351
Epoch: 600 Loss: 0.2669380620985963
Epoch: 700 Loss: 0.17439536298489544
Epoch: 800 Loss: 0.12017459805892042
Epoch: 900 Loss: 0.08736926423389581


## Predict Function

In [10]:
def predict(X_input):

    Z1 = np.dot(X_input, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    confidence = np.max(A2)

    if confidence < 0.75:
        return "Unknown Person"

    predicted_class = np.argmax(A2, axis=1)[0]

    return num_to_label[predicted_class]

## Test New Image

In [13]:
test_image_path = "test/pic1.jfif"

test_image = Image.open(test_image_path).convert('L')

test_image = test_image.resize((32, 32))

test_image = np.array(test_image).flatten()

test_image = test_image / 255.0

test_image = test_image.reshape(1, -1)

prediction = predict(test_image)

print("Recognized Person:", prediction)

Recognized Person: Babar
